# BigAlpha 2026 - Mamba End-to-End Stock Prediction
- Params: ~626K | Fields: 24 | Lookback: 20 days | 5-min bars
- Architecture: Patch Embed -> Mamba x4 -> Attention Pool -> MLP Head
- SEED=42, pure PyTorch, reproducible

In [ ]:
import os, gc, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch

from config_mamba import (
    MambaConfig, set_seed, SEED,
    SELECTED_FIELDS, FREQ_TABLES, set_active_freqs
)
from mamba_model import MambaStockModel, build_model
from data_pipeline import DataPipeline, query_all_freqs

set_seed(SEED)
print(f"PyTorch {torch.__version__}")


# ============================================================
# JSON 权重加载 (内联, 不依赖 train_mamba.py 的 scipy 等训练专用库)
# ============================================================
def load_model_json(model_path, map_location="cpu"):
    """读取 JSON 权重文件, 按 dtype/shape 还原张量。

    返回: {"state_dict": {name: Tensor}, "config": {...}, ...}
    """
    with open(model_path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    sd = {}
    for k, meta in payload["state_dict"].items():
        t = torch.tensor(meta["data"], dtype=getattr(torch, meta["dtype"]))
        sd[k] = t.reshape(meta["shape"]).to(map_location)
    ckpt = {k: v for k, v in payload.items() if k != "state_dict"}
    ckpt["state_dict"] = sd
    return ckpt

In [ ]:
def main(datasources, start_date, end_date):
    """平台调用入口。

    推理不查开发表算统计量 — 统计量已在本地训练时算好存 JSON。
    平台只需: 加载权重 → 查注入表 → 归一化 → 推理。
    """
    config = MambaConfig()
    set_active_freqs(config.freq_names)
    device = config.device
    print(f"Device: {device} | freq: {config.freq_names}")

    # ---- 推理表: 取平台注入的 ----
    infer_table = datasources.get("bar5m")
    if infer_table is None:
        raise KeyError(f"datasources 缺少 'bar5m' 键, 可用的: {list(datasources.keys())}")
    print(f"Infer table: {infer_table}")

    # ---- 加载 JSON 权重 + 统计量 + 配置 ----
    model_path = None
    for f in sorted(os.listdir(".")):
        if f.endswith(".json") and f != "norm_stats.json":
            model_path = f
            break
    if model_path is None:
        raise FileNotFoundError("未找到 .json 权重文件")
    print(f"Model: {model_path}")

    ckpt = load_model_json(model_path, map_location=device)
    saved_cfg = ckpt.get("config", {})
    config.n_fields     = saved_cfg.get("n_fields", config.n_fields)
    config.lookback_days = saved_cfg.get("lookback_days", config.lookback_days)
    config.bars_per_day = saved_cfg.get("bars_per_day", config.bars_per_day)
    config.seq_len      = config.lookback_days * config.bars_per_day
    config.d_model      = saved_cfg.get("d_model", config.d_model)
    config.n_layers     = saved_cfg.get("n_layers", config.n_layers)
    config.d_state      = saved_cfg.get("d_state", config.d_state)
    config.d_conv       = saved_cfg.get("d_conv", config.d_conv)
    config.patch_len    = saved_cfg.get("patch_len", config.patch_len)
    config.patch_stride = saved_cfg.get("patch_stride", config.patch_stride)

    # 从 JSON 读取本地预计算的归一化统计量 (免去平台端 DB 查询)
    norm_stats = ckpt.get("norm_stats", {})
    # JSON: {"open": [mean, std], ...} → {field: (mean, std)}
    field_stats = {k: (v[0], v[1]) for k, v in norm_stats.items()}

    model, n_params = build_model(config, verbose=False)
    model.load_state_dict(ckpt["state_dict"])
    model = model.to(device).eval()
    val_sign = ckpt.get("val_sign", 1)
    val_metrics = ckpt.get("val_metrics", {})
    print(f"Params: {n_params:,} | Norm fields: {len(field_stats)} "
          f"| val_sign: {val_sign}")

    # ---- 交易日历: 直接生成，不查 DB ----
    s_dt = pd.Timestamp(start_date)
    e_dt = pd.Timestamp(end_date)
    buf_dt = s_dt - pd.Timedelta(days=config.lookback_days + 10)
    all_trading_days = pd.bdate_range(buf_dt, e_dt).tolist()
    day_to_idx = {d: i for i, d in enumerate(all_trading_days)}
    print(f"Calendar: {len(all_trading_days)} business days ({buf_dt.date()} ~ {e_dt.date()})")

    # ---- 数据管道: 设统计量, 不调 fit() ----
    pipeline = DataPipeline(
        freq_names=config.freq_names,
        lookback_days=config.lookback_days,
        normalization=config.normalization,
        use_log_volume=config.use_log_volume)
    pipeline.stats = {config.freq_names[0]: field_stats}
    pipeline.all_trading_days = all_trading_days
    pipeline.day_to_idx = day_to_idx

    # ---- 推理查询 & 索引化 & 预测 ----
    import time as _time
    import dai as _dai
    fields_str = ", ".join(SELECTED_FIELDS)

    eval_dates = [d for d in all_trading_days if s_dt <= d <= e_dt]
    print(f"Eval: {len(eval_dates)} days")

    CHUNK_DAYS, all_rows, chunk_start = 10, [], 0
    while chunk_start < len(eval_dates):
        chunk_end = min(chunk_start + CHUNK_DAYS, len(eval_dates))
        chunk_dates = eval_dates[chunk_start:chunk_end]
        load_start = chunk_dates[0] - pd.Timedelta(days=config.lookback_days + 5)
        load_end = chunk_dates[-1]
        cn = chunk_start // CHUNK_DAYS + 1

        # 查询注入表 (日期只用日期部分, 避免 Timestamp 含时间导致格式重复)
        t0 = _time.time()
        df_infer = _dai.query(
            f"SELECT date, instrument, {fields_str} FROM {infer_table} "
            f"ORDER BY date, instrument",
            filters={"date": [f"{load_start.strftime('%Y-%m-%d')} 00:00:00",
                            f"{load_end.strftime('%Y-%m-%d')} 23:59:59"]},
            compression=True,
        ).df()
        print(f"  Chunk {cn}: {load_start.date()}~{load_end.date()} | "
              f"query {len(df_infer):,}r in {_time.time()-t0:.0f}s")

        raw = {config.freq_names[0]: df_infer}
        indexed = pipeline.index_data(raw)
        del raw, df_infer
        gc.collect()

        freq0 = config.freq_names[0]
        for date_obj in chunk_dates:
            di = day_to_idx.get(date_obj)
            if di is None:
                continue
            ds = date_obj.strftime("%Y-%m-%d")
            insts = sorted([inst for inst in indexed[freq0] if ds in indexed[freq0][inst]])
            if not insts:
                continue

            for start in range(0, len(insts), 500):
                batch = insts[start:start+500]
                try:
                    B, D, T, F = len(batch), config.lookback_days, config.bars_per_day, config.n_fields
                    X = np.zeros((B, D*T, F), dtype=np.float32)
                    valid = np.zeros(B, dtype=np.bool_)
                    lb_start_idx = max(0, di - D + 1)
                    lb_dates = all_trading_days[lb_start_idx:di + 1]
                    lb_strs = [pd.Timestamp(d).strftime("%Y-%m-%d") for d in lb_dates]
                    offset = D - len(lb_dates)

                    for i, inst in enumerate(batch):
                        inst_dict = indexed[freq0].get(inst)
                        if inst_dict is None:
                            continue
                        has_data = False
                        for d_idx, d_str in enumerate(lb_strs):
                            out_day = offset + d_idx
                            if out_day < 0 or out_day >= D:
                                continue
                            tup = inst_dict.get(d_str)
                            if tup is None:
                                continue
                            tensor, mask = tup
                            if tensor.shape[0] == 0:
                                continue
                            nb = min(tensor.shape[1], T)
                            X[i, out_day*T:out_day*T+nb, :] = tensor[0, :nb, :]
                            has_data = True
                        valid[i] = has_data

                    if valid.sum() < 10:
                        continue
                    X_t = torch.from_numpy(X[valid]).to(device)
                    with torch.no_grad():
                        scores = model(X_t)
                        if val_sign == -1:
                            scores = -scores
                    for j, idx in enumerate(np.where(valid)[0]):
                        sc = scores[j].item()
                        if not np.isnan(sc):
                            all_rows.append({"date": ds, "instrument": batch[idx], "score": sc})
                    del X_t
                except Exception:
                    pass

        del indexed
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        chunk_start = chunk_end

    df = pd.DataFrame(all_rows, columns=["date", "instrument", "score"])
    df = df.sort_values(["date", "instrument"]).reset_index(drop=True)
    df.to_csv("scores.csv", index=False)
    print(f"Done: {len(df):,} rows | {df['date'].nunique()}d | {df['instrument'].nunique()} stocks")
    return df

In [ ]:
# ============================================================
# 本地调试入口 — 提交时平台不会执行此 cell,
# 平台会 import notebook 然后调用 main(datasources, start_date, end_date)
# ============================================================
if __name__ == "__main__":
    # 模拟平台注入: 本地用开发表名, 平台会替换为公榜/私榜对应表
    datasources = {"bar5m": "bigalpha_2026_stock_bar5m"}
    start_date = "2024-01-01 00:00:00"
    end_date   = "2024-12-31 23:59:59"
    print(f"本地调试: {start_date} ~ {end_date}")
    score_data = main(datasources, start_date, end_date)
    print("\n分数预览:")
    print(score_data.head(10))
    print(f"\n统计: mean={score_data['score'].mean():.4f} std={score_data['score'].std():.4f}")